# Week 03 - solvePnP Pose

2D-3D correspondence와 OpenCV solvePnP로 pose를 추정하고 이미지 위에 좌표축을 그리는 노트북.

## Day 1: 2D-3D Correspondence 이해

* 목표

    * PnP가 2D 이미지 점과 3D object point의 대응으로 pose를 구한다는 것을 이해합니다.

* 할 일

    * object point와 image point 개념 정리
    * object frame 기준 3D 점 정의 방식 정리
    * image coordinate 기준 2D 점 정의 방식 정리
    * correspondence 순서가 왜 중요한지 정리
    * `solvePnP` 입력/출력 형식 확인

* 산출물

    * `docs/week3_pnp_correspondence.md`


## 0. 오늘의 목표

오늘의 목표는 PnP가 무엇을 입력으로 받아서 pose를 추정하는지 이해하는 것이다.

PnP는 `Perspective-n-Point`의 약자로, 여러 개의 3D 점과 그 점들이 이미지에서 보이는 2D 점의 대응 관계를 이용해 물체의 pose를 구하는 방법이다.

6D Pose Estimation 관점에서 PnP는 다음 단계에 해당한다.

```text
Camera Calibration
-> 2D-3D Correspondence 준비
-> solvePnP
-> rvec, tvec 추정
-> T_co 구성
-> 3D axis projection으로 확인
```

즉, Week 3에서는 카메라 내부 파라미터 `K`를 이미 알고 있다고 가정하고, 이미지 위의 2D 점과 물체 기준 3D 점을 연결해서 object-to-camera pose를 구한다.

---

## 1. PnP가 풀고 싶은 문제

PnP가 풀고 싶은 문제는 다음과 같다.

```text
이미 알고 있는 것:
- 물체 위 기준점들의 3D 좌표
- 이미지에서 그 기준점들이 찍힌 2D 픽셀 좌표
- camera intrinsic matrix K
- distortion coefficients

구하고 싶은 것:
- 물체 좌표계에서 카메라 좌표계로 가는 rotation
- 물체 좌표계에서 카메라 좌표계로 가는 translation
```

수식으로 쓰면 다음과 같다.

```text
P_c = R_co P_o + t_co
```

여기서 `P_o`는 object frame 기준 3D point이고, `P_c`는 camera frame 기준 3D point이다.

PnP는 직접 `P_c`를 관측하는 것이 아니라, `P_o`가 이미지 위의 어느 픽셀 `(u, v)`에 보이는지를 이용해서 `R_co`, `t_co`를 추정한다.

---

## 2. 2D-3D Correspondence란?

`2D-3D correspondence`는 같은 물리적 점을 3D 좌표와 2D 이미지 좌표로 짝지은 것이다.

예를 들어 큐브의 한 꼭짓점이 있다고 하자.

```text
3D object point:
P_o[0] = [0, 0, 0]

2D image point:
p_img[0] = [320, 240]
```

이 둘은 같은 물리적 점을 의미한다.

차이는 표현되는 좌표계이다.

```text
P_o[0]     : object frame 기준 3D 좌표
p_img[0]  : image coordinate 기준 2D 픽셀 좌표
```

따라서 correspondence는 다음처럼 생각하면 된다.

```text
object_points[0] <-> image_points[0]
object_points[1] <-> image_points[1]
object_points[2] <-> image_points[2]
object_points[3] <-> image_points[3]
```

중요한 점은 인덱스가 같으면 같은 물리적 점이어야 한다는 것이다.

---

## 3. Object Point란?

`object point`는 물체 좌표계, 즉 object frame 기준으로 정의한 3D 점이다.

예를 들어 정사각형 마커의 한 변 길이가 `L`이고, 마커 중심을 object frame origin으로 잡는다면 네 꼭짓점은 다음처럼 정의할 수 있다.

```python
object_points = np.array([
    [-L/2,  L/2, 0],
    [ L/2,  L/2, 0],
    [ L/2, -L/2, 0],
    [-L/2, -L/2, 0],
], dtype=np.float32)
```

여기서 각 점은 이미지에서 보이는 점이 아니라, 실제 물체 위의 기준점이다.

즉, object point는 카메라가 어디에 있든 변하지 않는다.

```text
object point = 물체 자체에 고정된 3D 기준점
```

6D pose에서 object point가 중요한 이유는, pose를 추정할 때 “이 물체의 어느 3D 점이 이미지의 어느 2D 점으로 보였는가?”를 알아야 하기 때문이다.

---

## 4. Image Point란?

`image point`는 object point가 이미지 위에서 관측된 2D 픽셀 좌표이다.

예를 들어 마커의 네 꼭짓점이 이미지에서 다음 픽셀에 보였다고 하자.

```python
image_points = np.array([
    [250, 180],
    [390, 175],
    [400, 320],
    [245, 330],
], dtype=np.float32)
```

각 점은 image coordinate 기준이다.

```text
u: 이미지의 가로 방향 픽셀 좌표
v: 이미지의 세로 방향 픽셀 좌표
```

일반적인 OpenCV 이미지 좌표계에서는 왼쪽 위가 원점이고, 오른쪽으로 갈수록 `u`가 증가하며, 아래로 갈수록 `v`가 증가한다.

따라서 image point는 다음 의미를 가진다.

```text
image point = 물체 위 3D 기준점이 이미지에서 보이는 2D 픽셀 위치
```

---

## 5. Correspondence 순서가 중요한 이유

PnP에서 가장 중요한 실수 포인트는 point order이다.

다음 두 배열이 있다고 하자.

```python
object_points = np.array([
    [-L/2,  L/2, 0],  # top-left
    [ L/2,  L/2, 0],  # top-right
    [ L/2, -L/2, 0],  # bottom-right
    [-L/2, -L/2, 0],  # bottom-left
], dtype=np.float32)

image_points = np.array([
    [250, 180],  # top-left
    [390, 175],  # top-right
    [400, 320],  # bottom-right
    [245, 330],  # bottom-left
], dtype=np.float32)
```

이 경우에는 인덱스별로 같은 점이 대응된다.

```text
object_points[0] <-> image_points[0] = top-left
object_points[1] <-> image_points[1] = top-right
object_points[2] <-> image_points[2] = bottom-right
object_points[3] <-> image_points[3] = bottom-left
```

하지만 image point 순서를 잘못 넣으면 solvePnP는 잘못된 대응을 맞다고 믿고 pose를 계산한다.

예를 들어 다음은 틀린 대응이다.

```text
object_points[0] = top-left
image_points[0]  = bottom-right
```

이 경우 코드는 에러 없이 실행될 수 있지만, 추정된 `rvec`, `tvec`는 물리적으로 이상한 pose가 될 수 있다.

따라서 PnP에서 correspondence 순서는 단순한 배열 문제가 아니라 pose 정확도를 결정하는 핵심 요소이다.

---

## 6. solvePnP의 입력과 출력

OpenCV의 `solvePnP`는 대략 다음 입력을 받는다.

```python
success, rvec, tvec = cv2.solvePnP(
    object_points,
    image_points,
    camera_matrix,
    dist_coeffs
)
```

각 입력의 의미는 다음과 같다.

```text
object_points:
- object frame 기준 3D 점들
- shape 예시: (N, 3)
- 단위: meter 또는 millimeter
- 물체 위 기준점 좌표

image_points:
- image coordinate 기준 2D 점들
- shape 예시: (N, 2)
- 단위: pixel
- object_points와 같은 순서여야 함

camera_matrix:
- camera intrinsic matrix K
- fx, fy, cx, cy 포함
- 3D camera point를 2D pixel로 projection할 때 사용

dist_coeffs:
- distortion coefficients
- 렌즈 왜곡 보정에 사용
- calibration 결과에서 얻음
```

출력은 다음과 같다.

```text
success:
- pose 추정 성공 여부

rvec:
- rotation vector
- object frame에서 camera frame으로 가는 회전을 나타냄
- cv2.Rodrigues로 rotation matrix R로 변환 가능

tvec:
- translation vector
- object frame origin이 camera frame에서 어디에 있는지를 나타냄
```

즉, `solvePnP`의 출력은 object-to-camera pose이다.

```text
rvec, tvec -> T_co
```

---

## 7. rvec, tvec와 T_co의 관계

`solvePnP`가 반환하는 `rvec`, `tvec`는 물체 좌표계의 점을 카메라 좌표계로 변환하는 pose를 나타낸다.

```text
P_c = R_co P_o + t_co
```

여기서 `rvec`는 `R_co`로 변환할 수 있다.

```python
R_co, _ = cv2.Rodrigues(rvec)
```

그 다음 `R_co`와 `tvec`를 합치면 4x4 transformation matrix `T_co`를 만들 수 있다.

```python
T_co = np.eye(4)
T_co[:3, :3] = R_co
T_co[:3, 3] = tvec.reshape(3)
```

이때 `T_co`의 의미는 다음과 같다.

```text
T_co: object frame 기준 point를 camera frame 기준 point로 변환하는 행렬
```

즉, Week 1에서 배운 좌표계 변환이 Week 3의 PnP 결과로 다시 등장한다.

---

## 8. 작은 예시: 마커 네 꼭짓점 correspondence

마커의 실제 크기를 알고 있고, 이미지에서 네 꼭짓점을 찾았다고 하자.

```python
import numpy as np

L = 0.05  # 5 cm marker

object_points = np.array([
    [-L/2,  L/2, 0],
    [ L/2,  L/2, 0],
    [ L/2, -L/2, 0],
    [-L/2, -L/2, 0],
], dtype=np.float32)

image_points = np.array([
    [250, 180],
    [390, 175],
    [400, 320],
    [245, 330],
], dtype=np.float32)
```

이 예시에서 반드시 확인해야 할 것은 숫자 자체가 아니라 순서이다.

```text
object_points[0]와 image_points[0]는 같은 물리적 꼭짓점이어야 한다.
object_points[1]와 image_points[1]는 같은 물리적 꼭짓점이어야 한다.
object_points[2]와 image_points[2]는 같은 물리적 꼭짓점이어야 한다.
object_points[3]와 image_points[3]는 같은 물리적 꼭짓점이어야 한다.
```

이 순서가 맞아야 solvePnP가 올바른 pose를 추정할 수 있다.

---

## 9. 오늘의 핵심 문장

PnP는 object frame 기준 3D 점과 image coordinate 기준 2D 점의 대응 관계를 이용해 object-to-camera pose를 추정하는 방법이다.

`object_points[i]`와 `image_points[i]`는 반드시 같은 물리적 점이어야 한다.

`solvePnP`의 출력인 `rvec`, `tvec`는 물체 좌표계의 점을 카메라 좌표계로 변환하는 `T_co`를 구성하는 데 사용된다.

따라서 PnP에서 가장 먼저 확인해야 할 것은 correspondence의 개수, 좌표계, 단위, 그리고 point order이다.

---

In [ ]:
import numpy as np

# =========================
# Week 3 Day 1
# 2D-3D Correspondence Check
# =========================

# 정사각형 마커 한 변 길이
# 단위는 meter로 가정
L = 0.05  # 5 cm

# 1. object frame 기준 3D 점 정의
# 마커 중심을 object frame origin으로 둔다고 가정
object_points = np.array([
    [-L/2,  L/2, 0.0],  # top-left
    [ L/2,  L/2, 0.0],  # top-right
    [ L/2, -L/2, 0.0],  # bottom-right
    [-L/2, -L/2, 0.0],  # bottom-left
], dtype=np.float32)

# 2. image coordinate 기준 2D 점 정의
# 지금은 예시 좌표이다.
# 실제 Week 3 Day 3에서는 이미지에서 직접 찍은 좌표로 바꾼다.
image_points = np.array([
    [250, 180],  # top-left
    [390, 175],  # top-right
    [400, 320],  # bottom-right
    [245, 330],  # bottom-left
], dtype=np.float32)

# 3. 각 점의 이름
point_names = [
    "top-left",
    "top-right",
    "bottom-right",
    "bottom-left"
]

# 4. shape 확인
print("object_points shape:", object_points.shape)
print("image_points shape:", image_points.shape)

# 5. correspondence 개수 확인
assert object_points.shape[0] == image_points.shape[0], "3D 점과 2D 점의 개수가 다릅니다."
assert object_points.shape[1] == 3, "object_points는 (N, 3) 형태여야 합니다."
assert image_points.shape[1] == 2, "image_points는 (N, 2) 형태여야 합니다."

# 6. correspondence 순서 확인 출력
print("\n2D-3D Correspondence")
print("--------------------")

for i, name in enumerate(point_names):
    print(f"{i}: {name}")
    print(f"   object point: {object_points[i]}")
    print(f"   image point : {image_points[i]}")

## Day 2: 3D Object Points 정의

* 목표

    * 큐브 또는 마커의 3D 기준점을 object frame에서 정의합니다.

* 할 일

    * object frame origin 결정
    * 큐브 또는 마커 corner point 정의
    * `object_points` 배열 생성
    * 단위 설정: meter 또는 millimeter
    * 점 순서 문서화

* 산출물

    * `notebooks/week3_day2_object_points.ipynb`
    * object point 좌표표

## Day 3: 2D Image Points 지정

* 목표

    * 이미지에서 object point와 대응되는 2D image point를 준비합니다.

* 할 일

    * 샘플 이미지 준비
    * 수동 또는 자동으로 2D corner point 선택
    * `image_points` 배열 생성
    * object point와 image point 순서 일치 확인
    * 대응점 시각화

* 산출물

    * image point가 표시된 이미지
    * `experiment_log.md`에 Experiment 003 기록


## Day 4: OpenCV solvePnP 실행

* 목표

    * `solvePnP`로 object-to-camera pose를 추정합니다.

* 할 일

    * camera matrix `K` 불러오기
    * distortion coefficient 불러오기
    * `cv2.solvePnP` 실행
    * `rvec`, `tvec` 출력
    * `rvec`를 rotation matrix `R`로 변환
    * `T_co` 구성

* 산출물

    * `notebooks/week3_day4_solvepnp.ipynb`
    * 추정된 `rvec`, `tvec`, `T_co`

## Day 5: 3D Axis Projection

* 목표

    * 추정한 pose가 맞는지 이미지 위에 3D 좌표축을 그려 확인합니다.

* 할 일

    * object frame의 x, y, z축 점 정의
    * `cv2.projectPoints` 사용
    * 이미지 위에 3D axis overlay
    * 축 방향이 예상과 맞는지 확인
    * 이상하면 좌표계 방향, point order, camera matrix 확인

* 산출물

    * 3D axis overlay 이미지
    * pose 결과 해석 노트

## Day 6: RANSAC과 Reprojection Error 확인

* 목표

    * 잘못된 correspondence가 pose에 미치는 영향을 이해하고 RANSAC을 적용합니다.

* 할 일

    * `cv2.solvePnPRansac` 실행
    * inlier/outlier 확인
    * reprojection error 계산
    * 일반 `solvePnP`와 RANSAC 결과 비교
    * 실패 케이스 기록

* 산출물

    * RANSAC 결과
    * reprojection error 비교
    * `error_log.md` 업데이트

## Day 7: 정리 및 README 업데이트

* 목표

    * 3주차 PnP 실습 결과를 README와 실험노트에 정리합니다.

* 할 일

    * Experiment 003 정리
    * `object_points`, `image_points` 설명 추가
    * `solvePnP` 입력/출력 정리
    * `rvec`, `tvec`, `T_co` 의미 정리
    * 3D axis overlay 이미지 추가
    * 다음 주 RGB-D / Point Cloud로 넘어갈 준비

* 산출물

    * `README.md` 업데이트
    * `experiment_log.md` 업데이트
    * `project_state.md` 업데이트